# Indices de lexicalité

In [ ]:
pip install wordfreq

In [ ]:
import os
import re
import unicodedata
from collections import Counter
from wordfreq import zipf_frequency, top_n_list

# -----------------------------
# Normalization & Tokenization
# -----------------------------

def normalize_text(text):
    text = text.lower()
    text = ''.join(
        c for c in unicodedata.normalize('NFD', text) # décomposition des caractère avec diacritiques puis suppression des accents
        if unicodedata.category(c) != 'Mn'
    )
    return text


def tokenize(text):
    return re.findall(r"\b[a-z]+\b", text)

# -----------------------------
# Word Validation
# -----------------------------

def word_score(word, lang="en"):
    return zipf_frequency(word, lang) # fréquence du mot dans la langue


def is_valid_word(word, threshold=2.5):
    return word_score(word) >= threshold # les mots rares ne sont pas considérés

# -----------------------------
# Language Reference
# -----------------------------

LANG_TOP_WORDS_CACHE = {}

def get_language_top_words(lang="en", n=100):
    if (lang, n) not in LANG_TOP_WORDS_CACHE:
        words = top_n_list(lang, n * 2)  # sur-échantillonnage

        # normalisation cohérente avec ton tokenizer
        words = [
            w.lower()
            for w in words
            if re.match(r'^[a-z]+$', w)
        ]

        LANG_TOP_WORDS_CACHE[(lang, n)] = set(words[:n])

    return LANG_TOP_WORDS_CACHE[(lang, n)]

# -----------------------------
# OCR Analysis
# -----------------------------

def analyze_text(text, threshold=2.5, lang="en"):
    raw_text = text
    text = normalize_text(text)

    # -----------------------------
    # Caractères globaux
    # -----------------------------
    non_space_chars = len(re.findall(r"\S", text))
    digit_chars = len(re.findall(r"\d", text))
    alpha_chars = len(re.findall(r"[a-z]", text))
    other_chars = non_space_chars - digit_chars - alpha_chars

    # -----------------------------
    # Tokenisation
    # -----------------------------
    words = tokenize(text)
    total_words = len(words)

    if total_words == 0:
        return None

    chars_in_words = sum(len(w) for w in words)

    removed_chars = non_space_chars - chars_in_words
    removed_ratio = removed_chars / non_space_chars if non_space_chars else 0

    digit_ratio = digit_chars / non_space_chars if non_space_chars else 0
    other_ratio = other_chars / non_space_chars if non_space_chars else 0

    # -----------------------------
    # Scores lexicaux
    # -----------------------------
    scores = [word_score(w, lang) for w in words]

    valid_words = [w for w in words if is_valid_word(w, threshold)]
    invalid_words = [w for w in words if not is_valid_word(w, threshold)]

    valid_chars = sum(len(w) for w in valid_words)
    lexical_char_ratio = valid_chars / non_space_chars if non_space_chars else 0

    lexicality = len(valid_words) / total_words * 100

    # -----------------------------
    # Top words intersection
    # -----------------------------
    word_counts = Counter(words)
    top_ocr_words = set([w for w, _ in word_counts.most_common(100)])

    lang_top_words = get_language_top_words(lang, 100)

    intersection_size = len(top_ocr_words & lang_top_words)
    intersection_ratio = intersection_size / 100

    # -----------------------------
    # Résultat
    # -----------------------------
    return {
        "total_words": total_words,
        "valid_words": len(valid_words),
        "invalid_words": len(invalid_words),

        "lexicality": lexicality,
        "avg_zipf": sum(scores) / len(scores),
        "median_zipf": sorted(scores)[len(scores)//2],

        # --- nouvelles métriques ---
        "non_space_chars": non_space_chars,
        "chars_in_words": chars_in_words,
        "removed_ratio": removed_ratio,

        "digit_ratio": digit_ratio,
        "other_ratio": other_ratio,

        "lexical_char_ratio": lexical_char_ratio,

        "top100_intersection": intersection_size,
        "top100_intersection_ratio": intersection_ratio,

        "top_unknown": Counter(invalid_words).most_common(20)
    }

# -----------------------------
# Folder Processing
# -----------------------------

def process_folder(root_dir, threshold=2.5, lang="en"):
    report = []

    for subdir, _, files in os.walk(root_dir):
        for file in files:
            if file.endswith(".ocr"):
                path = os.path.join(subdir, file)

                with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                    text = f.read()

                result = analyze_text(text, threshold, lang)
                if result:
                    result["file"] = path
                    report.append(result)

    return report

# -----------------------------
# Reporting
# -----------------------------

def compute_global_summary(report):
    if not report:
        return None

    total_words = sum(r["total_words"] for r in report)
    total_valid = sum(r["valid_words"] for r in report)

    total_chars = sum(r["non_space_chars"] for r in report)
    total_valid_chars = sum(r["lexical_char_ratio"] * r["non_space_chars"] for r in report)

    avg_removed = sum(r["removed_ratio"] for r in report) / len(report)
    avg_digits = sum(r["digit_ratio"] for r in report) / len(report)
    avg_noise = sum(r["other_ratio"] for r in report) / len(report)
    avg_intersection = sum(r["top100_intersection_ratio"] for r in report) / len(report)

    lexicality = (total_valid / total_words) * 100 if total_words else 0
    lexical_char_ratio = total_valid_chars / total_chars if total_chars else 0

    return {
        "total_files": len(report),
        "total_words": total_words,
        "lexicality": lexicality,
        "lexical_char_ratio": lexical_char_ratio,

        "avg_removed_ratio": avg_removed,
        "avg_digit_ratio": avg_digits,
        "avg_noise_ratio": avg_noise,

        "avg_top100_intersection": avg_intersection
    }


def print_report(report):
    for r in report:
        print("\n" + "="*60)
        print(f"File: {r['file']}")

        print(f"Lexicality: {r['lexicality']:.2f}%")
        print(f"Words: {r['valid_words']}/{r['total_words']}")

        print(f"Avg Zipf: {r['avg_zipf']:.2f}")
        print(f"Median Zipf: {r['median_zipf']:.2f}")

        print(f"Removed chars ratio: {r['removed_ratio']:.2%}")
        print(f"Digits ratio: {r['digit_ratio']:.2%}")
        print(f"Noise ratio: {r['other_ratio']:.2%}")
        print(f"Lexical char ratio (proportion de caractères appartenant à des mots valides): {r['lexical_char_ratio']:.2%}")

        print(f"Top100 intersection: {r['top100_intersection']}/100")

        print("Top unknown words:")
        for w, c in r['top_unknown']:
            print(f"  {w}: {c}")

# -----------------------------
# Main
# -----------------------------

def print_global_summary(summary):
    if not summary:
        print("No data to summarize")
        return

    print("\n" + "="*60)
    print("GLOBAL SUMMARY")
    print("#"*60)

    print(f"Total files: {summary['total_files']}")
    print(f"Total words: {summary['total_words']}")

    print(f"Lexicality (% de mots reconnus par wordfreq): {summary['lexicality']:.2f}%")
    print(f"Lexical char ratio (proportion de caractères appartenant à des mots valides): {summary['lexical_char_ratio']:.2%}")

    print(f"Avg removed chars (caractères ignorés par ton tokenizer (non [a-z]): {summary['avg_removed_ratio']:.2%}")
    print(f"Avg digit ratio (proportion de chiffres dans le texte): {summary['avg_digit_ratio']:.2%}")
    print(f"Avg noise ratio (caractères non lettres et non chiffres): {summary['avg_noise_ratio']:.2%}")

    print(f"Avg top100 intersection: {summary['avg_top100_intersection']:.2%}")


if __name__ == "__main__":
    ROOT_FOLDER = "istex-ocr"
    THRESHOLD = 2.5 # à moduler pour vérifier la présence du mot dans wordfreq
    LANG = "en"

    report = process_folder(ROOT_FOLDER, THRESHOLD, LANG)
    print_report(report)

    summary = compute_global_summary(report)
    print_global_summary(summary)

In [ ]:
import csv

def export_report_to_csv(report, output_path="report.csv"):
    if not report:
        print("No data to export")
        return

    # champs à exporter (on évite les structures complexes comme top_unknown)
    fieldnames = [
        "file",
        "total_words",
        "valid_words",
        "invalid_words",
        "lexicality",
        "avg_zipf",
        "median_zipf",
        "non_space_chars",
        "chars_in_words",
        "removed_ratio",
        "digit_ratio",
        "other_ratio",
        "lexical_char_ratio",
        "top100_intersection",
        "top100_intersection_ratio",
    ]

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for r in report:
            row = {k: r.get(k, None) for k in fieldnames}
            writer.writerow(row)

    print(f"CSV saved to: {output_path}")

In [ ]:
def export_summary_to_markdown(summary, output_path="summary.md"):
    if not summary:
        print("No summary to export")
        return

    md = f"""# OCR Analysis Summary

## Global Metrics

- **Total files**: {summary['total_files']}
- **Total words**: {summary['total_words']}

## Quality Metrics

- **Lexicality**: {summary['lexicality']:.2f}%
- **Lexical char ratio**: {summary['lexical_char_ratio']:.2%}

## Noise Metrics

- **Avg removed chars ratio**: {summary['avg_removed_ratio']:.2%}
- **Avg digit ratio**: {summary['avg_digit_ratio']:.2%}
- **Avg noise ratio**: {summary['avg_noise_ratio']:.2%}

## Language Consistency

- **Avg top 100 intersection**: {summary['avg_top100_intersection']:.2%}
"""

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(md)

    print(f"Markdown summary saved to: {output_path}")

In [ ]:
ROOT_FOLDER = "istex-ocr"
THRESHOLD = 2.5
LANG = "en"

report = process_folder(ROOT_FOLDER, THRESHOLD, LANG)

print_report(report)

summary = compute_global_summary(report)
print_global_summary(summary)

export_report_to_csv(report, "ocr_report.csv")
export_summary_to_markdown(summary, "ocr_summary.md")